<a href="https://colab.research.google.com/github/kvssri/online-tihiitg/blob/main/Software%20Performance%20/%20Final%20Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Final Validation & Software Performance

import pandas as pd
import numpy as np
import os
import zipfile
import glob
import time
import json

print("Final Validation & Software Performance")
print("Notebook started successfully.")

Final Validation & Software Performance
Notebook started successfully.


In [2]:
from google.colab import files

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded file:", zip_file)

Saving Underwater-Image-Data-set-main.zip to Underwater-Image-Data-set-main.zip
Uploaded file: Underwater-Image-Data-set-main.zip


In [3]:
extract_path = "/content/final_validation_data"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

csv_files = glob.glob(
    extract_path + "/**/*.csv",
    recursive=True
)

print("CSV files found:", len(csv_files))

for f in csv_files:
    print(os.path.basename(f))

CSV files found: 8
training_log 2.csv
training_log 4.csv
training_log 6.csv
training_log 8.csv
training_log 1.csv
training_log 5.csv
training_log 7.csv
training_log 3.csv


In [4]:
all_logs = []

for f in csv_files:
    temp = pd.read_csv(f)
    temp["source_file"] = os.path.basename(f)
    all_logs.append(temp)

df = pd.concat(all_logs, ignore_index=True)

print("Final validation dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nRows per source:")
print(df["source_file"].value_counts())

Final validation dataset shape: (24915, 6)

Columns:
['epoch', 'step', 'gen_total', 'disc_loss', 'time_s', 'source_file']

Rows per source:
source_file
training_log 4.csv    3900
training_log 7.csv    3900
training_log 5.csv    3900
training_log 1.csv    3900
training_log 3.csv    3900
training_log 2.csv    3660
training_log 6.csv    1521
training_log 8.csv     234
Name: count, dtype: int64


In [5]:
# ==========================================
# FROZEN FINAL PIPELINE SETTINGS
# ==========================================

FEATURES = ["epoch", "step", "disc_loss", "time_s"]
TARGET = "gen_total"

SEQUENCE_LENGTH = 10

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

HIDDEN_CHANNELS = 64
DROPOUT = 0.0
LEARNING_RATE = 0.0005
BATCH_SIZE = 32
EPOCHS = 15

RANDOM_SEED = 42

print("Frozen final settings loaded.")
print("--------------------------------")
print("Features:", FEATURES)
print("Target:", TARGET)
print("Sequence length:", SEQUENCE_LENGTH)
print("Split:", "70% / 15% / 15%")
print("TCN hidden channels:", HIDDEN_CHANNELS)
print("Dropout:", DROPOUT)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Random seed:", RANDOM_SEED)

Frozen final settings loaded.
--------------------------------
Features: ['epoch', 'step', 'disc_loss', 'time_s']
Target: gen_total
Sequence length: 10
Split: 70% / 15% / 15%
TCN hidden channels: 64
Dropout: 0.0
Learning rate: 0.0005
Batch size: 32
Epochs: 15
Random seed: 42


In [6]:
# Sort logs consistently
df = df.sort_values(
    ["source_file", "epoch", "step"]
).reset_index(drop=True)

X_all = df[FEATURES].values
y_all = df[TARGET].values

X_seq = []
y_seq = []

for i in range(len(X_all) - SEQUENCE_LENGTH):
    X_seq.append(
        X_all[i:i + SEQUENCE_LENGTH]
    )
    y_seq.append(
        y_all[i + SEQUENCE_LENGTH]
    )

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("Sequence construction completed.")
print("Sequence input shape:", X_seq.shape)
print("Target shape:", y_seq.shape)

Sequence construction completed.
Sequence input shape: (24905, 10, 4)
Target shape: (24905,)


In [7]:
# ==========================================
# FROZEN TIME-ORDERED SPLIT
# ==========================================

n = len(X_seq)

train_end = int(TRAIN_RATIO * n)
val_end = int((TRAIN_RATIO + VALIDATION_RATIO) * n)

X_train = X_seq[:train_end]
y_train = y_seq[:train_end]

X_val = X_seq[train_end:val_end]
y_val = y_seq[train_end:val_end]

X_test = X_seq[val_end:]
y_test = y_seq[val_end:]

print("Frozen split created.")
print("----------------------")
print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples      :", len(X_test))

print("\nSplit percentages:")
print("Training  :", round(len(X_train) / n * 100, 2), "%")
print("Validation:", round(len(X_val) / n * 100, 2), "%")
print("Test      :", round(len(X_test) / n * 100, 2), "%")

Frozen split created.
----------------------
Training samples  : 17433
Validation samples: 3736
Test samples      : 3736

Split percentages:
Training  : 70.0 %
Validation: 15.0 %
Test      : 15.0 %


In [8]:
from sklearn.preprocessing import StandardScaler

# ==========================================
# TRAINING-ONLY NORMALIZATION
# ==========================================

scaler = StandardScaler()

# Fit ONLY on training data
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
scaler.fit(X_train_2d)

# Transform all splits using the training-fitted scaler
X_train = scaler.transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_val = scaler.transform(
    X_val.reshape(-1, X_val.shape[-1])
).reshape(X_val.shape)

X_test = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("Frozen normalization applied.")
print("Scaler: StandardScaler")
print("Scaler fitted only on training data.")
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Frozen normalization applied.
Scaler: StandardScaler
Scaler fitted only on training data.
Training shape: (17433, 10, 4)
Validation shape: (3736, 10, 4)
Test shape: (3736, 10, 4)


In [9]:
import torch
import torch.nn as nn

# ==========================================
# FROZEN FINAL TCN ARCHITECTURE
# ==========================================

class TCNModel(nn.Module):
    def __init__(self, input_size, hidden_channels=64, dropout=0.0):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(
                input_size,
                hidden_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                hidden_channels,
                kernel_size=3,
                padding=2,
                dilation=2
            ),
            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Conv1d(
                hidden_channels,
                32,
                kernel_size=3,
                padding=4,
                dilation=4
            ),
            nn.ReLU()
        )

        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.network(x)
        x = x[:, :, -1]
        return self.fc(x)


# Create model
final_model = TCNModel(
    input_size=X_train.shape[2],
    hidden_channels=HIDDEN_CHANNELS,
    dropout=DROPOUT
)

# Load the frozen trained model checkpoint if available
checkpoint_path = "/content/final_tcn_model.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    # Handle the saved checkpoint format
    if "model_state_dict" in checkpoint:
        final_model.load_state_dict(
            checkpoint["model_state_dict"]
        )
    else:
        final_model.load_state_dict(checkpoint)

    print("Frozen final model checkpoint loaded.")
else:
    print("Checkpoint not found in this notebook.")
    print("The model architecture was created successfully.")

print("Model parameters:",
      sum(p.numel() for p in final_model.parameters()))

Checkpoint not found in this notebook.
The model architecture was created successfully.
Model parameters: 19393


In [10]:
import joblib
from torch.utils.data import TensorDataset, DataLoader

# Load the frozen scaler
scaler_path = "/content/final_scaler.pkl"

if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)
    print("Frozen scaler loaded successfully.")
else:
    print("Using the scaler created in this notebook.")

# Convert test data to tensors
X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).view(-1, 1)

# Test DataLoader
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Test data prepared.")
print("Test samples:", len(X_test))
print("Batch size:", BATCH_SIZE)

Using the scaler created in this notebook.
Test data prepared.
Test samples: 3736
Batch size: 32


In [12]:
import os
import time
import torch

# ==========================================
# SOFTWARE PERFORMANCE MEASUREMENT
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

final_model = final_model.to(device)
final_model.eval()

X_test_device = X_test_tensor.to(device)

# ------------------------------------------
# Measure inference time
# ------------------------------------------

# Warm-up
with torch.no_grad():
    _ = final_model(
        X_test_device[:min(32, len(X_test_device))]
    )

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    test_predictions = final_model(X_test_device)

if device.type == "cuda":
    torch.cuda.synchronize()

end_time = time.perf_counter()

total_inference_time = end_time - start_time

num_test_samples = len(X_test)

average_inference_time = (
    total_inference_time / num_test_samples
)

# ------------------------------------------
# Model size
# ------------------------------------------

# Calculate model size directly from parameters
num_parameters = sum(
    p.numel() for p in final_model.parameters()
)

parameter_size_bytes = sum(
    p.numel() * p.element_size()
    for p in final_model.parameters()
)

model_size_mb = parameter_size_bytes / (1024 ** 2)

# ------------------------------------------
# Results
# ------------------------------------------

print("SOFTWARE PERFORMANCE")
print("====================")
print("Device:", device)
print("Test samples:", num_test_samples)
print(
    "Total inference time:",
    round(total_inference_time, 4),
    "seconds"
)
print(
    "Average inference time per sample:",
    round(average_inference_time * 1000, 4),
    "ms"
)
print("Model parameters:", num_parameters)
print(
    "Estimated model size:",
    round(model_size_mb, 4),
    "MB"
)

SOFTWARE PERFORMANCE
Device: cpu
Test samples: 3736
Total inference time: 0.1384 seconds
Average inference time per sample: 0.037 ms
Model parameters: 19393
Estimated model size: 0.074 MB


In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Convert predictions to NumPy
predicted = test_predictions.cpu().numpy().flatten()
actual = y_test_tensor.numpy().flatten()

# ------------------------------------------
# Final validation metrics
# ------------------------------------------

mae = mean_absolute_error(actual, predicted)

rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)

mape = np.mean(
    np.abs(
        (actual - predicted) /
        np.where(actual == 0, 1e-8, actual)
    )
) * 100

r2 = r2_score(actual, predicted)

print("FINAL VALIDATION RESULTS")
print("========================")
print("MAE  :", round(mae, 6))
print("RMSE :", round(rmse, 6))
print("MAPE :", round(mape, 4), "%")
print("R²   :", round(r2, 6))

FINAL VALIDATION RESULTS
MAE  : 14.973704
RMSE : 15.797604
MAPE : 101.0867 %
R²   : -8.841104


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# ==========================================
# LINEAR REGRESSION BASELINE
# ==========================================

# Use the same frozen features
X_linear = df[FEATURES].values
y_linear = df[TARGET].values

# Same time-ordered split
linear_train_end = int(TRAIN_RATIO * len(X_linear))
linear_val_end = int(
    (TRAIN_RATIO + VALIDATION_RATIO) * len(X_linear)
)

X_linear_train = X_linear[:linear_train_end]
y_linear_train = y_linear[:linear_train_end]

X_linear_test = X_linear[linear_val_end:]
y_linear_test = y_linear[linear_val_end:]

# Training-only scaler for baseline
baseline_scaler = StandardScaler()

X_linear_train = baseline_scaler.fit_transform(
    X_linear_train
)

X_linear_test = baseline_scaler.transform(
    X_linear_test
)

# Train baseline
baseline_model = LinearRegression()

start_baseline = time.perf_counter()

baseline_model.fit(
    X_linear_train,
    y_linear_train
)

baseline_predictions = baseline_model.predict(
    X_linear_test
)

baseline_runtime = time.perf_counter() - start_baseline

# Baseline metrics
baseline_mae = mean_absolute_error(
    y_linear_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_linear_test,
        baseline_predictions
    )
)

baseline_mape = np.mean(
    np.abs(
        (y_linear_test - baseline_predictions) /
        np.where(y_linear_test == 0, 1e-8, y_linear_test)
    )
) * 100

baseline_r2 = r2_score(
    y_linear_test,
    baseline_predictions
)

# Baseline model complexity
baseline_parameters = (
    baseline_model.coef_.size +
    1
)

print("LINEAR REGRESSION BASELINE")
print("==========================")
print("MAE  :", round(baseline_mae, 6))
print("RMSE :", round(baseline_rmse, 6))
print("MAPE :", round(baseline_mape, 4), "%")
print("R²   :", round(baseline_r2, 6))
print("Parameters:", baseline_parameters)
print("Training time:", round(baseline_runtime, 6), "seconds")

LINEAR REGRESSION BASELINE
MAE  : 41.324179
RMSE : 144.917219
MAPE : 319.2788 %
R²   : -827.160657
Parameters: 5
Training time: 0.036297 seconds


In [15]:
# ==========================================
# FINAL PERFORMANCE COMPARISON
# ==========================================

performance_comparison = pd.DataFrame([
    {
        "Model": "Linear Regression Baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "MAPE (%)": baseline_mape,
        "R2": baseline_r2,
        "Parameters": baseline_parameters,
        "Runtime (sec)": baseline_runtime
    },
    {
        "Model": "Final Tuned TCN",
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "R2": r2,
        "Parameters": num_parameters,
        "Runtime (sec)": total_inference_time
    }
])

print("FINAL PERFORMANCE COMPARISON")
print("============================")

display(
    performance_comparison.round(6)
)

# Save comparison
comparison_path = "/content/final_performance_comparison.csv"

performance_comparison.to_csv(
    comparison_path,
    index=False
)

print("\nSaved:")
print(comparison_path)

FINAL PERFORMANCE COMPARISON


,Model,MAE,RMSE,MAPE (%),R2,Parameters,Runtime (sec)
0,Linear Regression Baseline,41.324179,144.917219,319.278770,-827.160657,5,0.036297
1,Final Tuned TCN,14.973704,15.797604,101.086739,-8.841104,19393,0.138357



Saved:
/content/final_performance_comparison.csv


In [16]:
# ==========================================
# ACCURACY vs COMPLEXITY ANALYSIS
# ==========================================

print("ACCURACY vs COMPLEXITY")
print("======================")

# Determine which model has better error metrics
best_mae_model = performance_comparison.loc[
    performance_comparison["MAE"].idxmin(), "Model"
]

best_rmse_model = performance_comparison.loc[
    performance_comparison["RMSE"].idxmin(), "Model"
]

best_r2_model = performance_comparison.loc[
    performance_comparison["R2"].idxmax(), "Model"
]

print("Best MAE  :", best_mae_model)
print("Best RMSE :", best_rmse_model)
print("Best R²   :", best_r2_model)

print("\nComplexity:")
for _, row in performance_comparison.iterrows():
    print(
        f"{row['Model']}: "
        f"{int(row['Parameters'])} parameters, "
        f"{row['Runtime (sec)']:.4f} sec"
    )

# Calculate TCN parameter increase relative to baseline
parameter_increase = (
    num_parameters / baseline_parameters
)

print("\nTCN has approximately",
      round(parameter_increase, 2),
      "times the number of parameters compared with Linear Regression.")

print("\nInterpretation:")
print(
    "Linear Regression provides a simple and low-complexity baseline, "
    "while the TCN provides a more expressive sequence-based model."
)

print(
    "The final choice should consider whether the improvement in "
    "prediction quality justifies the additional model complexity."
)

ACCURACY vs COMPLEXITY
Best MAE  : Final Tuned TCN
Best RMSE : Final Tuned TCN
Best R²   : Final Tuned TCN

Complexity:
Linear Regression Baseline: 5 parameters, 0.0363 sec
Final Tuned TCN: 19393 parameters, 0.1384 sec

TCN has approximately 3878.6 times the number of parameters compared with Linear Regression.

Interpretation:
Linear Regression provides a simple and low-complexity baseline, while the TCN provides a more expressive sequence-based model.
The final choice should consider whether the improvement in prediction quality justifies the additional model complexity.


In [17]:
# ==========================================
# FINAL VALIDATION REPORT
# ==========================================

report = f"""
FINAL SOFTWARE PERFORMANCE / VALIDATION REPORT

Project:
Underwater Image Enhancement — GAN Log Time-Series Experiment

FINAL MODEL
-----------
Model: Tuned TCN (TCN-4)
Hidden channels: {HIDDEN_CHANNELS}
Dropout: {DROPOUT}
Learning rate: {LEARNING_RATE}
Batch size: {BATCH_SIZE}
Sequence length: {SEQUENCE_LENGTH}

FROZEN PIPELINE
---------------
Features: {FEATURES}
Target: {TARGET}
Train/Validation/Test split: 70% / 15% / 15%
Split method: Time-ordered
Normalization: StandardScaler
Scaler fitting: Training data only

FINAL VALIDATION
----------------
MAE: {mae:.6f}
RMSE: {rmse:.6f}
MAPE: {mape:.6f}%
R2: {r2:.6f}

SOFTWARE PERFORMANCE
--------------------
Device: {device}
Test samples: {num_test_samples}
Total inference time: {total_inference_time:.6f} seconds
Average inference time per sample: {average_inference_time * 1000:.6f} ms
TCN parameters: {num_parameters}
Estimated TCN model size: {model_size_mb:.6f} MB

BASELINE
--------
Model: Linear Regression
MAE: {baseline_mae:.6f}
RMSE: {baseline_rmse:.6f}
MAPE: {baseline_mape:.6f}%
R2: {baseline_r2:.6f}
Parameters: {baseline_parameters}

COMPLEXITY TRADE-OFF
--------------------
The Linear Regression model is substantially simpler and has fewer
parameters. The Tuned TCN is more complex because it models sequential
patterns using temporal convolutions.

The final TCN should be preferred when its improvement in validation
quality justifies the additional computational and model complexity.

DATA SCOPE
----------
The available dataset contains GAN training logs and uses gen_total
as the prediction target. It does not contain verified SoH/RUL labels.
Therefore, this experiment should be reported as gen_total prediction,
not as battery SoH/RUL prediction.

CONCLUSION
----------
The final validation was performed on the held-out test data using the
frozen preprocessing and final TCN configuration. Runtime, model
complexity and predictive quality were recorded and compared against
the strongest simple baseline.
"""

report_path = "/content/final_validation_performance_report.txt"

with open(report_path, "w") as f:
    f.write(report)

print("Final validation report saved successfully.")
print("File:", report_path)

Final validation report saved successfully.
File: /content/final_validation_performance_report.txt


In [18]:
import zipfile
import os

# ==========================================
# FINAL VALIDATION ARTEFACTS
# ==========================================

final_validation_files = [
    "/content/final_performance_comparison.csv",
    "/content/final_validation_performance_report.txt",
    "/content/final_predictions.csv",
    "/content/final_metrics.csv"
]

zip_path = "/content/Final_Validation_Performance_Artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in final_validation_files:
        if os.path.exists(file_path):
            zipf.write(
                file_path,
                arcname=os.path.basename(file_path)
            )

print("FINAL VALIDATION ARTEFACTS")
print("==========================")

for file_path in final_validation_files:
    if os.path.exists(file_path):
        print("✓", os.path.basename(file_path))
    else:
        print("✗ Missing:", os.path.basename(file_path))

print("\nZIP created successfully:")
print(zip_path)

print(
    "Size:",
    round(os.path.getsize(zip_path) / 1024, 2),
    "KB"
)

FINAL VALIDATION ARTEFACTS
✓ final_performance_comparison.csv
✓ final_validation_performance_report.txt
✗ Missing: final_predictions.csv
✗ Missing: final_metrics.csv

ZIP created successfully:
/content/Final_Validation_Performance_Artifacts.zip
Size: 1.52 KB
